# CS383: Data Science and Machine Learning
## Lecture 2 Exercises -- Python Refresher, NumPy, Vectorized Computing

Fill in every `__________` blank, then run all cells top to bottom. When you've completed this
notebook, download it (File -> Save and Export Notebook As -> Notebook (.ipynb), or the **Download**
button in the toolbar) and submit it on BrightSpace under **Lecture 2 Exercise** as a Jupyter Notebook
(.ipynb) file.

In [ ]:
import numpy as np

---

## Exercise 1 -- List Comprehensions and `zip()`

### Scenario
You have two parallel lists: NYC boroughs and their approximate populations. A list comprehension lets you build a new list from an existing one in a single line, instead of writing an explicit loop with `.append()`.

In [ ]:
boroughs = ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"]
populations = [1_630_000, 2_590_000, 2_270_000, 1_360_000, 490_000]

### Step 1 -- The loop way (given)

In [ ]:
populations_millions_loop = []
for p in populations:
    populations_millions_loop.append(p / 1_000_000)

print(populations_millions_loop)

### Step 2 -- Your turn: rewrite it as a list comprehension

Fill in the blank so `populations_millions` computes the exact same thing as `populations_millions_loop`, in a single line.

In [ ]:
populations_millions = [__________ for p in populations]

print(populations_millions)
print(f"Match: {populations_millions == populations_millions_loop}")

### Step 3 -- Your turn: pair two lists with `zip()`

Write a list comprehension that pairs `boroughs` with `populations` using `zip()`, keeping only the boroughs with a population over 2,000,000. Fill in both blanks.

In [ ]:
large_boroughs = [b for b, p in zip(__________, __________) if p > 2_000_000]

print(large_boroughs)

### Quick check
- Run `list(zip(boroughs, populations))` by itself in a scratch cell -- what does `zip()` actually produce?
- Could you write Step 3's comprehension using `range(len(boroughs))` instead of `zip()`? Which version is easier to read?

---

## Exercise 2 -- Dictionaries and Safe Access with `.get()`

### Scenario
Real API records -- like the NYC 311 complaints you've been pulling since Lecture 1 -- arrive as dictionaries, and real-world data is never perfectly complete. The sample records below are missing a field or two, on purpose.

In [ ]:
sample_complaints = [
    {"complaint_type": "Noise - Residential", "borough": "BROOKLYN", "agency": "NYPD"},
    {"complaint_type": "Illegal Parking", "agency": "NYPD"},        # no borough
    {"complaint_type": "HEAT/HOT WATER", "borough": "BRONX"},        # no agency
    {"borough": "QUEENS", "agency": "DOT"},                          # no complaint_type
]

### Step 1 -- Why `record["borough"]` is risky

The second record above has no `"borough"` key at all. `record["borough"]` would raise a `KeyError` on that record -- Python has no way to guess what value you wanted instead. This is exactly why every 311 and restaurant-inspection data pull in this course reaches for `.get()` instead of `[...]`, just like you saw in Lecture 2.

### Step 2 -- Your turn: access safely with `.get()`

Fill in the blank so the loop below prints each record's borough, using `"UNKNOWN"` as a default whenever the field is missing.

In [ ]:
for record in sample_complaints:
    borough = record.__________("borough", "UNKNOWN")
    print(borough)

### Step 3 -- Your turn: count what's missing

Using `.get()` and a comprehension (just like Exercise 1!), count how many records are missing an `"agency"` key. Fill in the blank -- think about what `.get("agency")` returns when the key doesn't exist versus when it does.

In [ ]:
missing_agency_count = sum(1 for r in sample_complaints if r.get("agency") __________ None)

print(f"{missing_agency_count} record(s) missing an agency")

### Quick check
- What would `record.get("borough")` return for a record missing that key, if you don't pass a default at all?
- Why is `record.get("agency") is None` a safer check than `record["agency"] == None`?

---

## Exercise 3 -- NumPy Array Basics

### Scenario
Before diving into vectorized calculations in the next two exercises, get comfortable creating, inspecting, and filtering NumPy arrays -- the building blocks every vectorized operation depends on.

### Step 1 -- Your turn: create an array

Fill in the blank to turn this list of daily complaint totals into a NumPy array.

In [ ]:
daily_complaint_totals = __________([412, 389, 501, 476, 288, 210, 340])

print(daily_complaint_totals)
print(type(daily_complaint_totals))

### Step 2 -- Your turn: inspect it

Fill in the three blanks to print this array's shape, data type, and total number of elements.

In [ ]:
print(daily_complaint_totals.__________)   # dimensions
print(daily_complaint_totals.__________)   # data type
print(daily_complaint_totals.__________)   # total element count

### Step 3 -- Your turn: index and slice

Fill in the blanks: get the first day's total, the last day's total, and the middle three days' totals (index 2 through 4, inclusive).

In [ ]:
first_day = daily_complaint_totals[__________]
last_day = daily_complaint_totals[__________]
middle_three = daily_complaint_totals[__________]

print(first_day, last_day, middle_three)

### Step 4 -- Your turn: boolean mask

Fill in the blank so `busy_days` contains only the days where the complaint total was above 400.

In [ ]:
busy_days = daily_complaint_totals[__________]

print(busy_days)
print(f"{len(busy_days)} day(s) had more than 400 complaints")

### Quick check
- What would `daily_complaint_totals > 400` print if you ran it by itself, without putting it inside `[...]`?
- Try `np.arange(0, 7)` in a scratch cell -- how is it different from `daily_complaint_totals`?

---

## Exercise 4 — Vectorized Math vs. Python Loops

In this lab you'll write both a loop version and a vectorized version of the same calculation, time them yourself, and see the difference firsthand.

### Scenario
You have exam scores for a large class and want to compute each student's *squared deviation* from the class average (`(score - mean_score) ** 2`) — the same building block used to compute variance and standard deviation, which you'll see formalized later in the course.

In [ ]:
rng = np.random.default_rng(383)
scores = rng.normal(loc=75, scale=10, size=200_000).round(1)
scores = np.clip(scores, 0, 100)   # keep scores in a valid 0-100 range

print(scores[:10])
print(len(scores))

### Step 1 — Loop version

In [ ]:
import time

start = time.time()
mean_score = sum(scores) / len(scores)
squared_devs_loop = []
for s in scores:
    squared_devs_loop.append((s - mean_score) ** 2)
loop_time = time.time() - start

print(f"Loop version took {loop_time:.3f} seconds")
print(squared_devs_loop[:5])

### Step 2 — Your turn: vectorize it

Fill in the two blanks so `squared_devs_vectorized` computes the same thing as `squared_devs_loop`, without writing a Python `for` loop.

In [ ]:
start = time.time()
mean_score_np = scores.mean()
squared_devs_vectorized = (__________ - __________) ** 2
vectorized_time = time.time() - start

print(f"Vectorized version took {vectorized_time:.5f} seconds")
print(squared_devs_vectorized[:5])
print(f"Speedup: {loop_time / vectorized_time:,.0f}x")
print(f"Results match: {np.allclose(squared_devs_loop, squared_devs_vectorized)}")

### Reflect
- How much faster was your vectorized version?
- What would happen to the loop version's time if you doubled `size` to 400,000 in the data cell above? Try it and see if your prediction was right.

---

## Exercise 5 — Basic Data Calculations, on Real Data

Same idea as Part 5, now applied to the live NYC 311 dataset from Lecture 1.

In [ ]:
import os
import pandas as pd

try:
    raw_path = os.path.expanduser("~/shared/nyc311_snapshot.csv")
    rows = pd.read_csv(raw_path).head(20000).to_dict("records")
    complaint_arr = np.array([r.get("complaint_type", "UNKNOWN") for r in rows])
    borough_arr = np.array([r.get("borough", "UNKNOWN") for r in rows])
    live = True

except Exception:
    # Offline fallback, in case there is no internet connection in the room.
    rng = np.random.default_rng(383)
    complaint_types = ["Noise - Residential", "Illegal Parking", "HEAT/HOT WATER",
                        "Blocked Driveway", "Street Condition", "Water System",
                        "PAINT/PLASTER", "Damaged Tree", "Sewer", "Rodent"]
    boroughs = ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"]
    complaint_arr = rng.choice(
        complaint_types, size=20000,
        p=[0.18, 0.15, 0.14, 0.10, 0.10, 0.09, 0.08, 0.06, 0.05, 0.05],
    )
    borough_arr = rng.choice(boroughs, size=20000, p=[0.22, 0.32, 0.26, 0.16, 0.04])
    live = False

print(f"{'Shared snapshot' if live else 'Offline fallback'} data: {len(complaint_arr):,} 311 records")
print(complaint_arr[:5])

### Counting, the loop way

In [ ]:
target = "Noise - Residential"

start = time.time()
count_loop = 0
for c in complaint_arr:
    if c == target:
        count_loop += 1
loop_time = time.time() - start

print(f"'{target}' appeared {count_loop:,} times")
print(f"Loop took {loop_time:.4f} seconds")

### Your turn: count the vectorized way

Fill in both blanks so `count_vectorized` counts the same thing as `count_loop`, without a loop. The loop above compares each element to `target` one at a time with `c == target` -- a boolean mask does that same comparison across the whole array at once. What turns that array of `True`/`False` values into a single count? (Hint: think about what it means to *sum up* a bunch of `True`/`False` values.)

In [ ]:
start = time.time()
count_vectorized = __________(__________ == target)
vectorized_time = time.time() - start

print(f"Vectorized count: {count_vectorized:,}")
print(f"Vectorized took {vectorized_time:.6f} seconds")
print(f"Match: {count_loop == count_vectorized}")

### Basic summary calculations, fully vectorized

In [ ]:
# np.unique with return_counts=True gives back two arrays: the distinct borough
# names (sorted alphabetically) and how many times each one appears in borough_arr
boroughs_unique, borough_counts = np.unique(borough_arr, return_counts=True)

# Turn each borough's raw count into a percentage of the total records
borough_percentages = borough_counts / borough_counts.sum() * 100

# argsort gives the indices that would sort an array smallest to largest;
# negating borough_counts flips that to largest to smallest, so `order`
# holds the indices needed to go from most complaints to fewest
order = np.argsort(-borough_counts)

# Use `order` to pull all three arrays into that same most-to-fewest sequence,
# then print one formatted line per borough: name, count (with a thousands
# separator), and percentage rounded to one decimal place
for b, count, pct in zip(boroughs_unique[order], borough_counts[order], borough_percentages[order]):
    print(f"{b:15s} {count:6,d} complaints  ({pct:4.1f}%)")

`np.unique(..., return_counts=True)` is itself a vectorized operation — no loop required to tally every borough's complaint count.

---

## Exercise 6 — Reflection (Exit Ticket)

Answer the following in your own words.

1. What does it mean for a NumPy operation to be "vectorized"?
2. Why was the vectorized version of the exam-score calculation faster than the loop version?
3. What does `arr[arr > 20]` do, step by step?
4. Give one example (from today or otherwise) of a calculation where you'd still reach for a loop.
5. What question do you still have about NumPy or vectorization before Lecture 3?

**Your responses:**

1.  
2.  
3.  
4.  
5.  

## Optional Challenge

Pick any numeric data you care about (school grades, sports stats, workout logs, game scores, etc.). You'll build the full loop-vs-vectorized comparison one step at a time.

### Step 1 — Pick your data, write the loop version

In [ ]:
# At least 5 numeric values
my_data = [__________, __________, __________, __________, __________]

# Loop-based calculation: a sum, an average, or a count of values meeting some condition
# Your code here


### Step 2 — Rewrite it as a vectorized NumPy calculation

In [ ]:
# Convert my_data to a NumPy array, then redo Step 1's calculation without a loop
# Your code here


### Step 3 — Scale up and time both versions

In [ ]:
# Generate a much larger, randomly generated version of your data (at least 100,000 values)
# Time both the loop version and the vectorized version, and report the speedup
# Your code here


### Big idea
> The habit of asking "can this be vectorized?" instead of reaching for a loop first is one of the most valuable instincts you'll build this semester. It shows up again in Pandas starting Lecture 3, and in every ML model you train starting Week 6.